In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

In [ ]:
# Load training data
url = "https://huggingface.co/datasets/usaaio-official/2026_USAAIO_Round1_public/raw/main/2026_USAAIO_Round1_breast_cancer_train.csv"
df = pd.read_csv(url)
X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Standardize features - critical for kNN since it's distance-based
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Find optimal k using 5-fold cross-validation with macro F1 scoring
best_k, best_score = 1, 0
for k in range(1, 31):
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_scaled, y, cv=5, scoring='f1_macro')
    if scores.mean() > best_score:
        best_k, best_score = k, scores.mean()
print(f"Best k={best_k} with CV F1-macro={best_score:.4f}")

In [ ]:
# Train final model on all training data
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_scaled, y)

In [ ]:
def my_prediction(X_test):
    """Predict labels for test data using trained kNN model."""
    X_test_scaled = scaler.transform(X_test)
    return pd.Series(model.predict(X_test_scaled))

## Summary

**Approach:** Standard kNN classification with feature standardization and cross-validated hyperparameter tuning.

**Design choices:**
- **StandardScaler:** Essential for kNN since it uses Euclidean distance; features with larger scales would otherwise dominate.
- **k selection via CV:** Searched k=1-30 using 5-fold cross-validation optimizing macro F1 to match evaluation metric.
- **Simplicity:** No complex feature engineering since the 30 features are already numeric and kNN can handle moderate dimensionality.

**Alternatives considered:**
- PCA for dimensionality reduction - not used since 30 features is manageable and PCA may discard discriminative information.
- Different distance metrics (Manhattan, Minkowski) - Euclidean (default) works well for standardized data.
- Feature selection - not pursued to keep solution simple; standardization handles scale differences adequately.